# Практическое задание: DFS (Deep Feature Synthesis) на pandas

Аналог `featuretools.dfs(max_depth=2)` без сторонних зависимостей.
Работает на Python 3.12+.

## 1. Импорт и данные

In [ ]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "client_id": [1, 2, 3, 4],
    "name": ["Иванов", "Петров", "Сидоров", "Смирнов"],
    "join_date": pd.to_datetime(["2023-01-10", "2023-02-15", "2023-03-20", "2023-04-01"]),
    "region": ["МСК", "СПб", "МСК", "Казань"],
})
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "client_id": [1, 1, 2, 3, 1],
    "order_date": pd.to_datetime(["2023-04-01", "2023-05-05", "2023-05-10",
                                  "2023-06-01", "2023-06-10"]),
    "total_amount": [1500, 2300, 800, 3200, 1100],
})
products = pd.DataFrame({
    "product_id": [10, 20, 30],
    "category": ["электроника", "еда", "дом"],
    "price": [1000, 150, 2500],
})
order_items = pd.DataFrame({
    "item_id":    [1,   2,   3,   4,   5,   6],
    "order_id":   [101, 102, 103, 104, 105, 102],
    "product_id": [10,  20,  20,  30,  10,  30],
    "quantity":   [1,   2,   1,   1,   2,   1],
})

print("=== customers ===\n", customers)
print("\n=== orders ===\n", orders)
print("\n=== products ===\n", products)
print("\n=== order_items ===\n", order_items)

=== customers ===
    client_id     name  join_date  region
0          1   Иванов 2023-01-10     МСК
1          2   Петров 2023-02-15     СПб
2          3  Сидоров 2023-03-20     МСК
3          4  Смирнов 2023-04-01  Казань

=== orders ===
    order_id  client_id order_date  total_amount
0       101          1 2023-04-01          1500
1       102          1 2023-05-05          2300
2       103          2 2023-05-10           800
3       104          3 2023-06-01          3200
4       105          1 2023-06-10          1100

=== products ===
    product_id     category  price
0          10  электроника   1000
1          20          еда    150
2          30          дом   2500

=== order_items ===
    item_id  order_id  product_id  quantity
0        1       101          10         1
1        2       102          20         2
2        3       103          20         1
3        4       104          30         1
4        5       105          10         2
5        6       102          30    

## 2. «EntitySet» — словарь таблиц + связи (эмуляция ft.EntitySet)

In [ ]:
entityset = {
    "customers":   customers,
    "orders":      orders,
    "products":    products,
    "order_items": order_items,
}
relationships = [
    ("customers", "client_id",  "orders",      "client_id"),
    ("orders",    "order_id",   "order_items", "order_id"),
    ("products",  "product_id", "order_items", "product_id"),
]
print("Таблицы:", list(entityset.keys()))
print("Связи:", relationships)

Таблицы: ['customers', 'orders', 'products', 'order_items']
Связи: [('customers', 'client_id', 'orders', 'client_id'), ('orders', 'order_id', 'order_items', 'order_id'), ('products', 'product_id', 'order_items', 'product_id')]


## 3. Трансформационные примитивы (depth 0→1) — из собственных колонок customers

In [ ]:
c = customers.set_index("client_id")
feat = pd.DataFrame(index=c.index)

feat["region"]             = c["region"]
feat["MONTH(join_date)"]   = c["join_date"].dt.month
feat["YEAR(join_date)"]    = c["join_date"].dt.year
feat["WEEKDAY(join_date)"] = c["join_date"].dt.weekday
feat["DAY(join_date)"]     = c["join_date"].dt.day
feat["WEEK(join_date)"]    = c["join_date"].dt.isocalendar().week.astype(int)

## 4. Агрегации depth 1 — customers ← orders

In [ ]:
g1 = orders.groupby("client_id")

agg1 = pd.DataFrame({
    "SUM(orders.total_amount)":   g1["total_amount"].sum(),
    "MEAN(orders.total_amount)":  g1["total_amount"].mean(),
    "COUNT(orders)":              g1.size(),
    "MAX(orders.total_amount)":   g1["total_amount"].max(),
    "MIN(orders.total_amount)":   g1["total_amount"].min(),
    "STD(orders.total_amount)":   g1["total_amount"].std(),
    "NUNIQUE(orders.order_date)": g1["order_date"].nunique(),
    "MONTH(first_order)":         g1["order_date"].min().dt.month,
    "YEAR(last_order)":           g1["order_date"].max().dt.year,
    "WEEKDAY(last_order)":        g1["order_date"].max().dt.weekday,
})
feat = feat.join(agg1)

## 5. Агрегации depth 2 — customers ← orders ← order_items

Сначала агрегируем позиции внутри каждого заказа, затем поднимаем к клиенту.

In [ ]:
oi = order_items.merge(orders[["order_id", "client_id"]], on="order_id")

# depth-2 агрегаты по заказам
per_order = oi.groupby("order_id").agg(
    sum_qty=("quantity", "sum"),
    mean_qty=("quantity", "mean"),
    max_qty=("quantity", "max"),
    min_qty=("quantity", "min"),
    n_items=("item_id", "count"),
    n_unique_prod=("product_id", "nunique"),
    sum_price_weighted=("quantity", lambda x: 0),  # placeholder
).reset_index()

# обогащаем ценой продукта
oi_full = oi.merge(products, on="product_id")
per_order_cost = oi_full.groupby("order_id").apply(
    lambda d: (d["quantity"] * d["price"]).sum(), include_groups=False
).rename("order_cost")

# поднимаем к клиенту
g2 = oi.merge(per_order_cost.reset_index(), on="order_id").groupby("client_id")
per_client_items = oi.groupby("client_id")

agg2 = pd.DataFrame({
    "SUM(orders.SUM(order_items.quantity))":
        per_client_items["quantity"].sum().groupby(level=0).sum()
        if False else oi.groupby("client_id")["quantity"].sum(),
    "MEAN(orders.MEAN(order_items.quantity))":
        oi.merge(per_order[["order_id","mean_qty"]], on="order_id")
          .groupby("client_id")["mean_qty"].mean(),
    "MAX(orders.MAX(order_items.quantity))":
        oi.merge(per_order[["order_id","max_qty"]], on="order_id")
          .groupby("client_id")["max_qty"].max(),
    "SUM(orders.COUNT(order_items))":
        oi.groupby("client_id").size(),
    "MEAN(orders.NUNIQUE(order_items.product_id))":
        oi.groupby("client_id")["product_id"].nunique().groupby(level=0).mean()
        if False else oi.groupby(["client_id","order_id"])["product_id"].nunique()
                       .groupby(level=0).mean(),
    "SUM(orders.SUM(order_items.cost))":
        oi_full.groupby("client_id").apply(
            lambda d: (d["quantity"]*d["price"]).sum(), include_groups=False),
})
feat = feat.join(agg2)

## 6. Результат

In [ ]:
feat = feat.fillna(0)

print("Размер матрицы признаков:", feat.shape)
print(f"Всего признаков: {feat.shape[1]}\n")
print("Список признаков:")
for i, col in enumerate(feat.columns, 1):
    print(f"{i:>3}. {col}")

Размер матрицы признаков: (4, 22)
Всего признаков: 22

Список признаков:
  1. region
  2. MONTH(join_date)
  3. YEAR(join_date)
  4. WEEKDAY(join_date)
  5. DAY(join_date)
  6. WEEK(join_date)
  7. SUM(orders.total_amount)
  8. MEAN(orders.total_amount)
  9. COUNT(orders)
 10. MAX(orders.total_amount)
 11. MIN(orders.total_amount)
 12. STD(orders.total_amount)
 13. NUNIQUE(orders.order_date)
 14. MONTH(first_order)
 15. YEAR(last_order)
 16. WEEKDAY(last_order)
 17. SUM(orders.SUM(order_items.quantity))
 18. MEAN(orders.MEAN(order_items.quantity))
 19. MAX(orders.MAX(order_items.quantity))
 20. SUM(orders.COUNT(order_items))
 21. MEAN(orders.NUNIQUE(order_items.product_id))
 22. SUM(orders.SUM(order_items.cost))


In [ ]:
feat

,region,MONTH(join_date),YEAR(join_date),WEEKDAY(join_date),DAY(join_date),WEEK(join_date),SUM(orders.total_amount),MEAN(orders.total_amount),COUNT(orders),MAX(orders.total_amount),...,NUNIQUE(orders.order_date),MONTH(first_order),YEAR(last_order),WEEKDAY(last_order),SUM(orders.SUM(order_items.quantity)),MEAN(orders.MEAN(order_items.quantity)),MAX(orders.MAX(order_items.quantity)),SUM(orders.COUNT(order_items)),MEAN(orders.NUNIQUE(order_items.product_id)),SUM(orders.SUM(order_items.cost))
client_id,,,,,,,,,,,,,,,,,,,,,
1,МСК,1,2023,1,10,2,4900.0,1633.333333,3.0,2300.0,...,3.0,4.0,2023.0,5.0,6.0,1.5,2.0,4.0,1.333333,5800.0
2,СПб,2,2023,2,15,7,800.0,800.000000,1.0,800.0,...,1.0,5.0,2023.0,2.0,1.0,1.0,1.0,1.0,1.000000,150.0
3,МСК,3,2023,0,20,12,3200.0,3200.000000,1.0,3200.0,...,1.0,6.0,2023.0,3.0,1.0,1.0,1.0,1.0,1.000000,2500.0
4,Казань,4,2023,5,1,13,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
